# 🧠🤖 第2周-Day2：Tokenizer与词嵌入

🎯 **今日主题**：Tokenizer与词嵌入（BPE/SentencePiece）

📝 **昨日复习**：FFN、LayerNorm与残差连接
• FFN：前馈网络，提供非线性变换能力
• LayerNorm：层归一化，稳定训练过程
• 残差连接：解决梯度消失，让深层网络有效学习

🔑 **英文术语**：
- **Tokenization** [ˈtoʊkɪnaɪzeɪʃən] 分词
- **Vocabulary** [vəˈkæbjələri] 词汇表
- **Subword** [ˈsʌbwɜːrd] 子词
- **Embedding** [ɪmˈbɛdɪŋ] 词嵌入
- **BPE** [Bi-Pair Encoding] 字节对编码

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

## 📚 什么是分词？

分词是把连续的文本切分成有意义的单元（tokens）的过程。就像我们读句子时自然地分词一样，计算机也需要理解文本的结构。

💡 **生活关联**：就像糖水店记账时，我们把"红豆糖水"分成红豆、糖、水三个部分一样，分词也是把句子拆成有意义的小单元。

In [ ]:
# 简单的分词示例
text = "红豆糖水很好喝"
words = list(text)

print(f"原始文本: {text}")
print(f"分词结果: {words}")
print(f"词汇表大小: {len(set(words))}")

# 可视化
plt.figure(figsize=(10, 6))
word_counts = Counter(words)
plt.bar(word_counts.keys(), word_counts.values(), color='lightblue')
plt.title('字符级别分词统计')
plt.xlabel('字符')
plt.ylabel('出现次数')
plt.show()

## 🔤 词表（Vocabulary）

词表是模型能识别的所有token的集合。词表大小是Transformer模型的重要超参数：
• 小词表（如几千词）：节省内存，但可能丢失语义
• 大词表（如5万词）：更精确，但计算开销大

💡 **业务思考**：糖水店的菜单如果用小词表，"红豆沙"可能被分成"红豆"+"沙"，而大词表可以直接识别"红豆沙"这个完整概念。

In [ ]:
# 词表大小对模型的影响
vocab_sizes = [1000, 5000, 10000, 30000, 50000]
memory_usage = [size * 4 / 1024 / 1024 for size in vocab_sizes]  # MB
accuracy_estimate = [60, 75, 85, 92, 95]  # 模拟准确率

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 内存使用
ax1.plot(vocab_sizes, memory_usage, 'o-', color='red', linewidth=2, markersize=8)
ax1.set_xlabel('词表大小')
ax1.set_ylabel('内存使用 (MB)')
ax1.set_title('词表大小 vs 内存使用')
ax1.grid(True, alpha=0.3)

# 准确率
ax2.plot(vocab_sizes, accuracy_estimate, 'o-', color='blue', linewidth=2, markersize=8)
ax2.set_xlabel('词表大小')
ax2.set_ylabel('模拟准确率 (%)')
ax2.set_title('词表大小 vs 模型性能')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## ⚡ BPE (字节对编码)

BPE是Google提出的一种子词分词算法，特别擅长处理稀有词和未知词。它会合并最常见的字符对，逐步构建词表。

🎬 **推荐视频**：[BPE算法详解](https://www.bilibili.com/video/BV1w44y1f7Kv) (15分钟)

In [ ]:
class SimpleBPE:
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.vocab = {}
        self.merges = []
    
    def get_stats(self, vocab):
        """统计字符对频率"""
        pairs = {}
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] = pairs.get((symbols[i], symbols[i + 1]), 0) + freq
        return pairs
    
    def merge_vocab(self, pair, vocab_in):
        """合并字符对"""
        vocab_out = {}
        bigram = re.escape(' '.join(pair))
        p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
        for word in vocab_in:
            new_word = p.sub(''.join(pair), word)
            vocab_out[new_word] = vocab_in[word]
        return vocab_out
    
    def train(self, corpus):
        """训练BPE模型"""
        # 初始化：每个字符是一个token
        vocab = {}
        for word in corpus:
            vocab[' '.join(word)] = vocab.get(' '.join(word), 0) + 1
        
        print("初始词表:")
        print(list(vocab.keys())[:10])
        
        # 执行合并
        for i in range(self.num_merges):
            pairs = self.get_stats(vocab)
            if not pairs:
                break
            
            # 选择最频繁的对
            best_pair = max(pairs, key=pairs.get)
            vocab = self.merge_vocab(best_pair, vocab)
            self.merges.append(best_pair)
            
            print(f"步骤{i+1}: 合并 {best_pair} (频率: {pairs[best_pair]})")
        
        self.vocab = vocab
        return vocab

# 示例：糖水店相关文本
corpus = ["红豆", "糖水", "绿豆", "银耳", "莲子", "红豆", "糖水", "红豆", "莲子", "糖水"]

bpe = SimpleBPE(num_merges=5)
vocab = bpe.train(corpus)

print("\n最终词表:")
for word, freq in vocab.items():
    print(f"{word}: {freq}")

## 🧩 SentencePiece

SentencePiece是Google提出的一种subword分词器，特别适合多语言场景。它与BPE的主要区别：
• 不依赖空格分词（适合中文、日文等无空格语言）
• 使用统一的Unicode编码，支持多语言
• 更加鲁棒的tokenization过程

📖 **延伸阅读**：[SentencePaper论文解读](https://zhuanlan.zhihu.com/p/358368344)

In [ ]:
# SentencePiece vs 传统分词对比
def visualize_tokenization_methods():
    methods = ['字符级', '词级', 'BPE', 'SentencePiece']
    text = "红豆糖水很好喝"
    
    # 模拟不同方法的分词结果
    char_tokens = list(text)
    word_tokens = ["红豆", "糖水", "很好", "喝"]
    bpe_tokens = ["红豆", "糖", "水", "很", "好", "喝"]  # 简化示例
    sp_tokens = ["红豆", "糖", "水", "很", "好", "喝"]  # 简化示例
    
    plt.figure(figsize=(12, 8))
    
    # 原始文本
    plt.subplot(2, 2, 1)
    for i, char in enumerate(text):
        plt.annotate(char, (i, 0), ha='center', va='center', 
                    fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue"))
    plt.title('原始文本')
    plt.xlim(-0.5, len(text)-0.5)
    plt.ylim(-0.5, 0.5)
    plt.axis('off')
    
    # 字符级分词
    plt.subplot(2, 2, 2)
    for i, token in enumerate(char_tokens):
        plt.annotate(token, (i, 0), ha='center', va='center', 
                    fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgreen"))
    plt.title('字符级分词')
    plt.xlim(-0.5, len(char_tokens)-0.5)
    plt.ylim(-0.5, 0.5)
    plt.axis('off')
    
    # BPE分词
    plt.subplot(2, 2, 3)
    pos = 0
    for token in bpe_tokens:
        plt.annotate(token, (pos, 0), ha='center', va='center', 
                    fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"))
        pos += len(token)
    plt.title('BPE分词')
    plt.xlim(-0.5, pos-0.5)
    plt.ylim(-0.5, 0.5)
    plt.axis('off')
    
    # SentencePiece分词
    plt.subplot(2, 2, 4)
    pos = 0
    for token in sp_tokens:
        plt.annotate(token, (pos, 0), ha='center', va='center', 
                    fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightcoral"))
        pos += len(token)
    plt.title('SentencePiece分词')
    plt.xlim(-0.5, pos-0.5)
    plt.ylim(-0.5, 0.5)
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_tokenization_methods()

## 🔤 词嵌入（Embedding）

词嵌入是把离散的token转换为连续向量的技术，这是Transformer的核心创新之一：
• **One-hot编码**：稀疏向量，维度=词表大小
• **词嵌入**：稠密向量，通常维度为256-1024
• **语义相似性**：相似的词在向量空间中距离相近

💡 **AI关联**：就像大脑中"红豆"和"绿豆"的概念会激活相似的神经元群，词嵌入也让模型在数学层面理解词语的相似性。

In [ ]:
# 词嵌入可视化
def create_embedding_visualization():
    # 模拟词嵌入向量
    words = ['红豆', '绿豆', '银耳', '莲子', '糖', '水']
    
    # 生成2D嵌入向量（实际中通常是高维）
    np.random.seed(42)
    embeddings = np.random.randn(len(words), 2) * 0.5
    
    # 让相似词更接近
    embeddings[words.index('红豆')] = [-1, 0.5]  # 红豆
    embeddings[words.index('绿豆')] = [-0.8, 0.3]  # 绿豆（相似）
    embeddings[words.index('银耳')] = [0.8, -0.2]  # 银耳
    embeddings[words.index('莲子')] = [0.9, 0]  # 莲子（相似）
    embeddings[words.index('糖')] = [0, 1]  # 糖
    embeddings[words.index('水')] = [0, 0.8]  # 水（相似）
    
    # 绘制词嵌入空间
    plt.figure(figsize=(10, 8))
    
    # 绘制向量
    colors = ['red', 'green', 'orange', 'purple', 'pink', 'lightblue']
    for i, (word, embedding) in enumerate(zip(words, embeddings)):
        plt.scatter(embedding[0], embedding[1], c=colors[i], s=100, alpha=0.7)
        plt.annotate(word, (embedding[0], embedding[1]), fontsize=12, ha='center', va='center')
        
        # 绘制向量线
        plt.arrow(0, 0, embedding[0], embedding[1], head_width=0.05, 
                 head_length=0.05, fc=colors[i], ec=colors[i], alpha=0.5)
    
    # 绘制相似性连线
    plt.plot([-1, -0.8], [0.5, 0.3], 'k--', alpha=0.3, linewidth=1)  # 红豆-绿豆
    plt.plot([0.8, 0.9], [-0.2, 0], 'k--', alpha=0.3, linewidth=1)   # 银耳-莲子
    plt.plot([0, 0], [1, 0.8], 'k--', alpha=0.3, linewidth=1)       # 糖-水
    
    plt.title('词嵌入空间可视化', fontsize=14)
    plt.xlabel('维度1', fontsize=12)
    plt.ylabel('维度2', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.axis('equal')
    
    # 添加图例
    plt.figtext(0.02, 0.02, '虚线表示语义相似的词', fontsize=10, ha='left')
    
    plt.tight_layout()
    plt.show()

create_embedding_visualization()

## ✏️ 课堂练习（5分钟）

❶ **分词理解**：为什么中文不需要像英文那样明显的空格分词？
💡 提示：考虑汉字的特点和语言结构

❷ **词表大小**：如果词表大小是50,000，每个嵌入向量维度是768，计算：
   - 词嵌入矩阵的总参数量
   - 如果用one-hot编码，参数量会是多少倍？

❸ **BPE优势**：遇到"芒果千层蛋糕"这种新词，BPE为什么比词级分词更好？

❹ **语义相似性**："糖水"和"甜品"的词嵌入会相似吗？为什么？

❺ **实际应用**：糖水店顾客可能会说"红豆汤"而不是"红豆糖水"，这会对分词有什么影响？

## 📝 课后测试（15分钟）

❶ **选择题**：下列哪项不是子词分词的优势？
A. 处理未知词
B. 减少词表大小
C. 保留语义信息
D. 完全消除OOV问题

❷ **填空题**：BPE算法的步骤：1)初始化字符频率，2)统计______，3)合并最频繁的对，4)重复直到达到迭代次数。

❸ **判断题**：SentencePiece依赖空格进行分词，所以不适合中文。（ ）

❹ **简答题**：解释为什么"机器学习"比"机学习"更适合作为一个独立的token？

❺ **应用题**：假设我们要为糖水店建立一个专门的分词器，应该优先考虑哪些因素？请说明理由。

回复答案我帮你批改 ✅

## 🔄 往期回顾

🎯 **复习题**：Transformer中LayerNorm的作用是什么？如果移除LayerNorm，模型训练会出现什么问题？

💡 **提示**：考虑梯度流动和训练稳定性

## 📚 今日总结

🎯 **核心要点**：
• 分词是连接文本和模型的桥梁，不同语言需要不同策略
• 词表大小需要在精度和效率之间平衡
• BPE通过合并字符对处理未知词，SentencePiece更适合多语言
• 词嵌入将离散token转换为连续向量，捕捉语义相似性

🔗 **AI关联**：今天的分词和嵌入技术，实际上是让计算机像人类一样"理解"文字的开始，为后续的注意力机制打下基础。

💡 **业务思考**：糖水店AI助手如果分词不好，可能会把"红豆沙"误解为"红豆"+"沙"，影响用户体验。选择合适的分词器很重要！

📖 **预习建议**：明天我们将学习Transformer的整体架构，理解GPT和BERT的设计理念差异。